Inference is the process of loading a previously trained Machine Learning (ML) model and running it to calculate a prediction on new, unlabeled input data. It's the step where the model transitions from a research artifact to a piece of production software.

In [6]:
import pandas as pd
import pandas_datareader.data as web
import yfinance as yf
import pandas_ta_classic as ta
import numpy as np
import torch
from pytorch_forecasting import TemporalFusionTransformer, TimeSeriesDataSet
from datetime import datetime
from dateutil.relativedelta import relativedelta

# --- 1. CONFIGURATION ---
MODEL_PATH = "tft_model.ckpt"
CURRENCY_PAIR = 'EURUSD=X'
FRED_CODES = {
    'FEDFUNDS': 'US_Interest_Rate',
    'CPIAUCSL': 'US_Inflation',
    'UNRATE': 'US_Unemployment',
    'PAYEMS': 'US_NonFarm_Payrolls',
    'GDPC1': 'US_GDP',
    'BOPGSTB': 'US_Trade_Balance',
    'VIXCLS': 'Global_VIX',
    'DCOILWTICO': 'WTI_Crude_Oil_Price'
}
TARGET_COL = 'EURUSD=X' # Note: In the logic below, we rename this to Close_Price
FREQ = 'M'

class ForexForecaster:
    def __init__(self, model_path):
        print(f"Loading model from {model_path}...")
        # Load on CPU
        self.model = TemporalFusionTransformer.load_from_checkpoint(model_path, map_location=torch.device('cpu'))
        self.model.eval()
        
    def fetch_recent_data(self, lookback_months=60):
        """
        Fetches historical data and ensures strict column naming.
        """
        end_date = datetime.today()
        start_date = end_date - relativedelta(months=lookback_months)
        
        print(f"Fetching live data from {start_date.date()} to {end_date.date()}...")

        # 1. Fetch Forex (Robust Method)
        # auto_adjust=True removes the warning and handles splits/dividends (though rare for Forex)
        raw_yfinance = yf.download(CURRENCY_PAIR, start=start_date, end=end_date, interval='1d', progress=False, auto_adjust=True)
        
        # Handle yfinance MultiIndex columns (The root cause of your error)
        # Check if columns have levels (Price Type, Ticker)
        if isinstance(raw_yfinance.columns, pd.MultiIndex):
            try:
                # Try to extract 'Close'
                df_fx = raw_yfinance.xs('Close', level=0, axis=1)
            except KeyError:
                # Fallback if 'Close' isn't top level
                df_fx = raw_yfinance['Close']
        else:
            df_fx = raw_yfinance[['Close']]

        # Force valid DataFrame structure
        if isinstance(df_fx, pd.Series):
            df_fx = df_fx.to_frame()
            
        # If the column is still named strictly after the ticker, rename it
        # We assume the first column is the Close price
        df_fx = df_fx.iloc[:, [0]] 
        df_fx.columns = ["Close_Price"]
        
        # Remove Timezone info to match FRED data (FRED is usually naive)
        df_fx.index = df_fx.index.tz_localize(None)

        # 2. Fetch Macro (FRED)
        try:
            df_fred = web.DataReader(list(FRED_CODES.keys()), 'fred', start_date, end_date)
            df_fred.columns = list(FRED_CODES.values())
        except Exception as e:
            print(f"Warning: FRED fetch failed ({e}). Using zeros.")
            df_fred = pd.DataFrame(columns=FRED_CODES.values())

        # 3. Combine
        df = df_fx.join(df_fred, how='outer')
        
        # 4. Fill NaNs
        df = df.ffill().bfill()
        
        # Debug print to ensure column exists
        if 'Close_Price' not in df.columns:
            print("CRITICAL ERROR: 'Close_Price' column missing after join.")
            print(f"Columns found: {df.columns}")
        
        return df

    def preprocess_for_inference(self, df):
        df = df.copy()

        # Check for column existence before proceeding
        if 'Close_Price' not in df.columns:
            raise KeyError(f"'Close_Price' not found. Available columns: {df.columns.tolist()}")

        # --- A. Technical Indicators ---
        df['Lag_1D'] = df['Close_Price'].shift(1)
        df['SMA_20'] = ta.sma(df['Close_Price'], length=20)
        df['SMA_50'] = ta.sma(df['Close_Price'], length=50)
        df['RSI_14'] = ta.rsi(df['Close_Price'], length=14)
        df['Historical_Volatility_20D'] = df['Close_Price'].pct_change().rolling(window=20).std() * (252**0.5)

        # --- B. Resample to Monthly ---
        # Initialize resampled dataframe
        df_res = pd.DataFrame()
        
        # Resample Close_Price specifically
        df_res['EURUSD=X'] = df['Close_Price'].resample(FREQ).last()
        
        # Resample other columns
        # Note: We skip 'Close_Price' in the loop because we renamed it to 'EURUSD=X' above 
        # to match the Target name expected by the model during training.
        skip_cols = ['Close_Price']
        
        for col in df.columns:
            if col not in skip_cols:
                df_res[col] = df[col].resample(FREQ).last()
        
        df_res = df_res.ffill().dropna()

        # --- C. Feature Engineering (Lags) ---
        numeric_cols = df_res.select_dtypes(include=np.number).columns
        for col in numeric_cols:
            df_res[f'{col}_Lag1'] = df_res[col].shift(1)
            df_res[f'{col}_Lag4'] = df_res[col].shift(4)

        # --- D. Stationarity (Differencing) ---
        # Capture last state for reconstruction
        last_actual_price = df_res['EURUSD=X'].iloc[-1]
        last_actual_date = df_res.index[-1]

        # Apply differencing to non-lagged columns
        cols_to_diff = [c for c in df_res.columns if 'Lag' not in c] 
        
        for col in cols_to_diff:
            df_res[f'{col}_D1'] = df_res[col].diff()

        # Drop NaNs created by lags/diffs
        df_res = df_res.dropna()

        return df_res, last_actual_price, last_actual_date

    def predict_next_month(self):
        # 1. Get Data
        raw_df = self.fetch_recent_data()
        
        # 2. Process
        processed_df, last_price, last_date = self.preprocess_for_inference(raw_df)
        
        print(f"\nContext: Last closed month data available: {last_date.date()}")
        print(f"Context: Last Price: {last_price:.4f}")

        # 3. Prepare for PyTorch Forecasting
        # Create Dummy Row for next month to provide a target container
        new_row = processed_df.iloc[[-1]].copy()
        new_row.index = [last_date + relativedelta(months=1)]
        
        inference_df = pd.concat([processed_df, new_row])
        
        # Add required TFT columns
        inference_df['time_idx'] = np.arange(len(inference_df))
        inference_df['series'] = 'EURUSD'
        inference_df = inference_df.reset_index().rename(columns={'index': 'date'})

        # 4. Create Prediction Dataset
        try:
            # --- FIX IS HERE ---
            # Use 'from_parameters' instead of 'from_dataset' because dataset_parameters is a dict
            pred_ds = TimeSeriesDataSet.from_parameters(
                self.model.dataset_parameters, 
                inference_df, 
                predict=True, 
                stop_randomization=True
            )
            
            # Create dataloader
            pred_loader = pred_ds.to_dataloader(batch_size=1, shuffle=False)
            
            # 5. Predict
            # return_x=False returns only the predictions
            raw_prediction = self.model.predict(pred_loader, mode="prediction", return_x=False)
            
            # Extract the actual value (Batch 0, Step 0)
            predicted_change = raw_prediction[0][0].item() 
            
            # 6. Reconstruct Price
            predicted_price = last_price + predicted_change
            
            return {
                "target_date": (last_date + relativedelta(months=1)).date(),
                "last_price": last_price,
                "predicted_change": predicted_change,
                "predicted_price": predicted_price
            }

        except Exception as e:
            print(f"Inference Error: {e}")
            import traceback
            traceback.print_exc()
            return None
# --- EXECUTION ---
if __name__ == "__main__":
    print("--- starting inference pipeline ---")
    forecaster = ForexForecaster(MODEL_PATH)
    result = forecaster.predict_next_month()
    
    if result:
        print("\n" + "="*40)
        print(f"FORECAST REPORT FOR: {result['target_date'].strftime('%B %Y')}")
        print("="*40)
        print(f"Current Price:          {result['last_price']:.4f}")
        print(f"Predicted Change:       {result['predicted_change']:.4f}")
        print("-" * 40)
        print(f"FINAL PREDICTION:       {result['predicted_price']:.4f}")
        print("="*40)

--- starting inference pipeline ---
Loading model from tft_model.ckpt...


C:\Users\Vipin\anaconda3\envs\financialmodelingproject\lib\site-packages\lightning\pytorch\utilities\parsing.py:210: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
C:\Users\Vipin\anaconda3\envs\financialmodelingproject\lib\site-packages\lightning\pytorch\utilities\parsing.py:210: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.


Fetching live data from 2020-11-27 to 2025-11-27...


C:\Users\Vipin\AppData\Local\Temp\ipykernel_47972\3689960390.py:111: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df_res['EURUSD=X'] = df['Close_Price'].resample(FREQ).last()
C:\Users\Vipin\AppData\Local\Temp\ipykernel_47972\3689960390.py:120: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df_res[col] = df[col].resample(FREQ).last()
C:\Users\Vipin\AppData\Local\Temp\ipykernel_47972\3689960390.py:120: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df_res[col] = df[col].resample(FREQ).last()
C:\Users\Vipin\AppData\Local\Temp\ipykernel_47972\3689960390.py:120: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df_res[col] = df[col].resample(FREQ).last()
C:\Users\Vipin\AppData\Local\Temp\ipykernel_47972\3689960390.py:120: FutureWarning: 'M' is deprecated and will be removed in a 


Context: Last closed month data available: 2025-11-30
Context: Last Price: 1.1602


GPU available: False, used: False
TPU available: False, using: 0 TPU cores



FORECAST REPORT FOR: December 2025
Current Price:          1.1602
Predicted Change:       -0.0032
----------------------------------------
FINAL PREDICTION:       1.1570


C:\Users\Vipin\anaconda3\envs\financialmodelingproject\lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:433: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=17` in the `DataLoader` to improve performance.
